# 04 — Ensemble Fusion & Evaluation

Combines model scores, selects thresholds on validation, and evaluates on test — mirroring `main.py`.

Weights default to `ensemble_weights.json` (when present) or `config.yaml`. An optional section runs `optimise_weights()` for comparison.

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import json
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation import (
    evaluate_all_models,
    evaluate_all_threshold_methods,
    evaluate_per_attack_type,
)
from src.models import EnsembleDetector
from src.pipeline_helpers import (
    ensemble_weights_from_config,
    optimise_weights,
    apply_flow_ecod_fusion,
)
from src.visualization import plot_confusion_matrix
from nb_common import (
    NotebookSettings, load_project_config, load_dataframe,
    load_cached_splits, scale_splits, load_notebook_artifacts, setup_notebook,
)

SETTINGS = NotebookSettings(use_sample=True, use_cache=True)
cfg = load_project_config(SETTINGS)
setup_notebook(cfg)
RESULTS_DIR = SETTINGS.resolve_output_dir() / 'metrics'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Scores & Splits

In [2]:
artifacts = load_notebook_artifacts()
val_scores = artifacts['val_scores']
test_scores = dict(artifacts['test_scores'])
y_val, y_test = artifacts['y_val'], artifacts['y_test']
lab_test = artifacts['lab_test']
splits = load_cached_splits(SETTINGS, cfg)
MODELS_DIR = SETTINGS.resolve_output_dir() / 'models'
_, X_val, X_test, _ = scale_splits(splits, MODELS_DIR, load_existing=True)
print(f'Loaded {len(test_scores)} model score arrays')

Loaded 5 model score arrays


## 3. Ensemble (config / tuned weights)

In [3]:
weight_map = ensemble_weights_from_config(cfg)
print('Ensemble weights:')
for k, w in weight_map.items():
    print(f'  {k:20s} {w:.2f}')

ensemble_models = {k: None for k in weight_map}
ensemble = EnsembleDetector(models=ensemble_models, weights=weight_map)
val_scores = dict(val_scores)
val_scores['ensemble'] = ensemble.score_from({k: val_scores[k] for k in weight_map})
test_scores['ensemble'] = ensemble.score_from({k: test_scores[k] for k in weight_map})
print('Ensemble scores computed')

Ensemble weights:
  isolation_forest     0.25
  ecod                 0.75
Ensemble scores computed


## 4. Optional — Inline Weight Optimisation

In [4]:
individual_val = {k: v for k, v in val_scores.items() if k != 'ensemble'}
opt_w, opt_score = optimise_weights(
    individual_val, y_val, step=0.05, min_auroc=0.55, optimize_metric='auroc',
)
print('Optimised weights (validation AUROC grid):')
for k, w in sorted(opt_w.items()):
    if w > 0:
        print(f'  {k:20s} {w:.2f}')
print(f'Validation metric: {opt_score:.4f}')
print('\nFull hyperparameter tuning: python tune.py')

Optimised weights (validation AUROC grid):
  copod                0.05
  hbos                 0.95
Validation metric: 1.0000

Full hyperparameter tuning: python tune.py


## 5. Flow-Level ECOD Fusion (if enabled)

In [5]:
flow_w = float(cfg.get('models', {}).get('flow_ecod', {}).get('fusion_weight', 0.0))
if flow_w > 0 and not SETTINGS.use_sample:
    df = load_dataframe(SETTINGS, cfg)
    val_scores, test_scores, _ = apply_flow_ecod_fusion(
        df, cfg, val_scores, test_scores, y_val,
        artifacts['val_timestamps'], artifacts['test_timestamps'],
    )
    print(f'Applied flow ECOD fusion (weight={flow_w})')
else:
    print(f'Flow fusion skipped (fusion_weight={flow_w})')

Flow fusion skipped (fusion_weight=0.0)


## 6. Evaluation Metrics

In [6]:
eval_df = evaluate_all_models(
    test_scores, y_test, val_scores=val_scores, y_val=y_val,
)
display(eval_df)
eval_df.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)

,model,auroc,auprc,f1,precision,recall,fpr,fnr,threshold,threshold_method
0,vae,1.000000,1.000000,1.000000,1.000000,1.0,0.000000,0.0,1.000000,f1
1,isolation_forest,1.000000,1.000000,1.000000,1.000000,1.0,0.000000,0.0,1.000000,f1
2,ecod,1.000000,1.000000,1.000000,1.000000,1.0,0.000000,0.0,0.845865,f1
3,copod,1.000000,1.000000,1.000000,1.000000,1.0,0.000000,0.0,1.000000,f1
4,ensemble,1.000000,1.000000,1.000000,1.000000,1.0,0.000000,0.0,0.884399,f1
5,hbos,0.973684,0.941176,0.969697,0.941176,1.0,0.052632,0.0,1.000000,f1


## 7. Threshold Comparison

In [7]:
thr_methods = cfg.get('evaluation', {}).get(
    'threshold_methods', ['f1_optimal', 'youden_j', 'fpr_0.05', 'fpr_0.10']
)
thr_df = evaluate_all_threshold_methods(
    test_scores, y_test, val_scores, y_val, methods=thr_methods,
)
display(thr_df)
thr_df.to_csv(RESULTS_DIR / 'threshold_comparison.csv', index=False)

,model,threshold_method,auroc,auprc,f1,precision,recall,fpr,fnr,threshold
0,ensemble,f1_optimal,1.0,1.0,1.000000,1.0,1.000,0.0,0.000,0.884399
1,ensemble,youden_j,1.0,1.0,1.000000,1.0,1.000,0.0,0.000,0.884399
2,ensemble,fpr_0.05,1.0,1.0,0.933333,1.0,0.875,0.0,0.125,1.000000
3,ensemble,fpr_0.10,1.0,1.0,0.933333,1.0,0.875,0.0,0.125,0.994402


## 8. Per-Attack Metrics

In [8]:
thr_method = cfg.get('evaluation', {}).get('default_threshold_method', 'f1_optimal')
opt_thr, _ = EnsembleDetector.find_optimal_threshold(
    val_scores['ensemble'], y_val, method=thr_method,
)
per_attack = evaluate_per_attack_type(lab_test, test_scores['ensemble'], threshold=opt_thr)
if not per_attack.empty:
    display(per_attack)
    per_attack.to_csv(RESULTS_DIR / 'per_attack_metrics.csv', index=False)
print(f'Ensemble threshold (validation): {opt_thr:.4f}')

,attack_type,n_samples,auroc,auprc,f1,precision,recall,fpr
0,BOT,2,1.0,1.0,1.0,1.0,1.0,0.0
1,DDOS,4,1.0,1.0,1.0,1.0,1.0,0.0
2,FTP-PATATOR,2,1.0,1.0,1.0,1.0,1.0,0.0
3,PORTSCAN,5,1.0,1.0,1.0,1.0,1.0,0.0
4,SSH-PATATOR,3,1.0,1.0,1.0,1.0,1.0,0.0


Ensemble threshold (validation): 0.8844

## 9. Confusion Matrix

In [9]:
y_pred = (test_scores['ensemble'] >= opt_thr).astype(int)
fig = plot_confusion_matrix(y_test, y_pred)
plt.show()

## 10. Baseline Comparison (spectral vs raw)

In [10]:
import subprocess, sys
cmd = [sys.executable, 'scripts/baseline_comparison.py']
if SETTINGS.use_sample:
    cmd.append('--use-sample')
else:
    cmd.extend(['--data-dir', str(SETTINGS.resolve_data_dir())])
if SETTINGS.use_cache:
    cmd.append('--use-cache')
result = subprocess.run(cmd, cwd=str(SETTINGS.resolve_output_dir().parent), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
baseline_path = RESULTS_DIR / 'baseline_comparison.csv'
if baseline_path.is_file():
    display(pd.read_csv(baseline_path))

,config,feature_type,model,auroc,auprc,f1,precision,recall,fpr,fnr,threshold,threshold_method
0,spectral_ecod,spectral,ecod,1.000000,1.000000,1.000000,1.000000,1.000,0.000000,0.000,0.845865,f1
1,spectral_if,spectral,isolation_forest,0.973684,0.941176,0.969697,0.941176,1.000,0.052632,0.000,1.000000,f1
2,raw_ecod,raw_flow,ecod,0.996711,0.996324,0.933333,1.000000,0.875,0.000000,0.125,0.984033,f1
3,raw_if,raw_flow,isolation_forest,1.000000,1.000000,1.000000,1.000000,1.000,0.000000,0.000,1.000000,f1


## 11. Dual Evaluation (`any` label rule)

In [11]:
if cfg.get('evaluation', {}).get('dual_eval', False) and not SETTINGS.use_sample:
    from src.pipeline_helpers import build_ensemble_models, train_detectors, calibrate_all_detectors, score_all_models
    from src.preprocessor import DataPreprocessor
    from nb_common import build_splits
    any_cfg = deepcopy(cfg)
    any_cfg['evaluation']['window_label_rule'] = 'any'
    df = load_dataframe(SETTINGS, cfg)
    any_splits = build_splits(SETTINGS, any_cfg, df, cfg_override=any_cfg)
    pre = DataPreprocessor()
    Xtr = pre.fit_transform(any_splits['X_train'], feature_names=list(any_splits['feat_names']))
    Xva = pre.transform(any_splits['X_val'])
    Xte = pre.transform(any_splits['X_test'])
    m = build_ensemble_models(any_cfg, Xtr.shape[1])
    train_detectors(m, Xtr, any_cfg, X_val=Xva)
    calibrate_all_detectors(m, Xva, any_splits['y_val'])
    w = ensemble_weights_from_config(any_cfg)
    ens = EnsembleDetector(models={k: m[k] for k in w}, weights=w)
    te = score_all_models(m, Xte)
    te['ensemble'] = ens.score_from(te)
    per_any = evaluate_per_attack_type(any_splits['lab_test'], te['ensemble'])
    display(per_any)
else:
    print('Dual eval skipped (requires full CICIDS2017; set use_sample=False)')

Dual eval skipped (requires full CICIDS2017; set use_sample=False)
